<div style="font-size: 0.6em; max-width: 70ch">

# Importing packages

</div>

In [1]:
import pandas as pd
import plotly.express as px

# To display the figure defined by this dict, use the low-level 
# plotly.io.show function.
import plotly.io as pio 
# Default is plotly_mimetype+notebook, but jekyll fails to 
# parse plotly_mimetype.
pio.renderers.default = 'notebook_connected'
# Inject the missing require.js dependency.
from IPython.display import display, HTML
js = '<script src="https://cdnjs.cloudflare.com/ajax/libs/require.js' \
'/2.3.6/require.min.js" integrity="sha512-c3Nl8+7g4LMSTdrm621y7kf9v3SDP' \
'nhxLNhcjFJbKECVnmZHTdo+IRO05sNLTH/D3vA6u1X32ehoLC7WFVdheg==" ' \
'crossorigin="anonymous"></script>'
display(HTML(js))

import ardalan_proEV_functions as AF
# Import the custom functions for this notebook.
import cellgrowth.core as cg

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

<div style="font-size: 0.6em; max-width: 100ch">

# Importing and cleaning of the primary raw data

</div>

In [2]:
# Read the data file, skipping the first 8 rows of metadata.
df = pd.read_csv(r"../data/cell_growth_data.txt", 
                 delimiter = '\t', skiprows=8)

# Make a plate format inducating the row and column of the sample
# Map the numeric row values to letters for plate format.
cg.map_num_to_letter(df)

# Create a 'Plate format' column by concatenating the 
# 'Row' and 'Column' values.
df.Column = df.Column.astype(str)
df['Plate format'] = df[['Row', 'Column']].apply(lambda x: ''.join(x), axis=1)

In [3]:
df.head(1)

,Row,Column,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Temperature,Target Temperature,CO2,Target CO2,Compound,Concentration,Cell Type,Cell Count,Unnamed: 19,Plate format
0,E,4,0,770,302.658182,0.886863,2030.238556,1.044651e+07,768244.360521,25,0.0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,E4


In [4]:
df.columns.to_list()

['Row',
 'Column',
 'Timepoint',
 'Cells Selected - Number of Objects',
 'Cells Selected - Cell Area [µm²] - Mean per Well',
 'Cells Selected - Cell Roundness - Mean per Well',
 'Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well',
 'Image Region - Image Region Area [µm²] - Sum per Well',
 'Texture A Selected - Region Area [µm²] - Sum per Well',
 'Number of Analyzed Fields',
 'Time [s]',
 'Temperature',
 'Target Temperature',
 'CO2',
 'Target CO2',
 'Compound',
 'Concentration',
 'Cell Type',
 'Cell Count',
 'Unnamed: 19',
 'Plate format']

In [5]:
df = df [['Row',
 'Column',
 'Timepoint',
  'Plate format',
 'Cells Selected - Number of Objects',
 'Cells Selected - Cell Area [µm²] - Mean per Well',
 'Cells Selected - Cell Roundness - Mean per Well',
 'Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well',
 'Image Region - Image Region Area [µm²] - Sum per Well',
 'Texture A Selected - Region Area [µm²] - Sum per Well',
 'Number of Analyzed Fields',
 'Time [s]',
]]

<div style="font-size: 0.6em; max-width: 70ch">

# Data visualization

</div>

In [6]:
df.head(2)

,Row,Column,Timepoint,Plate format,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s]
0,E,4,0,E4,770,302.658182,0.886863,2030.238556,1.044651e+07,768244.360521,25,0.0
1,E,5,0,E5,881,322.810309,0.881544,2048.947241,1.044809e+07,701412.476601,25,0.0


In [7]:
df["Timepoint"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13],
      dtype=int64)

<div style="font-size: 0.6em; max-width: 70ch">

# Defining the samples of each well

</div>

In [8]:
df.loc[df['Plate format'].isin(['E4', 'E5']), ['Sample']] = 'Neg Control'

df.loc[df['Plate format'].isin(['F4', 'F5']), ['Sample']] = 'Parental'
 
df.loc[df['Plate format'].isin(['G4', 'G5']), ['Sample']] = 'LOW_uptake'
 
df.loc[df['Plate format'].isin(['H4', 'H5']), ['Sample']] = 'HIGH_uptake'

<div style="font-size: 0.6em; max-width: 100ch">

# Preparing dataframe for making lineplots

</div>

* As each sample is in duplicate, for making a line graph, one spot per time-point makes more sense. For that, I calculated the average of each
duplicate value using groupby. 

In [9]:
data = cg.DataProcessing(df)

# Process the timepoints to get a DataFrame with average time in seconds and 
# update the df inplace.  
# for each timepoint. This will be used for plotting the growth curves with 
# time on the x-axis.
timing = data.process_timepoints(
    timepoint_col='Timepoint', 
    time_col='Time [s]'
)

# Generate a grouped DataFrame for plotting growth curves.
df_grouped = data.grouped()
df_grouped.head()

,Sample,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Hours,cell_covered_area_std,cell_count_std
0,HIGH_uptake,0,1000.5,325.879734,0.872133,2297.879871,1.044785e+07,8.413457e+05,25.0,0.000,4.0,17480.305526,54.447222
1,HIGH_uptake,1,1056.5,374.740170,0.877443,2415.943798,1.044769e+07,1.087237e+06,25.0,21564.770,10.0,20712.256734,19.091883
2,HIGH_uptake,2,1032.0,480.300880,0.899131,2405.490523,1.044779e+07,1.444340e+06,25.0,43127.292,16.0,55226.688251,28.284271
3,HIGH_uptake,3,1180.0,542.115886,0.908378,2204.343429,1.044805e+07,1.935250e+06,25.0,70635.050,24.0,42149.396835,66.468037
4,HIGH_uptake,4,1426.0,544.635183,0.903243,2097.330607,1.044804e+07,2.314716e+06,25.0,91407.058,29.0,105368.804793,70.710678


"""Sort the samples for plotting:"""

In [10]:
# First check the order of the sample in the df, which determines their order 
# on the graph
df_grouped["Sample"].unique()

array(['HIGH_uptake', 'LOW_uptake', 'Neg Control', 'Parental'],
      dtype=object)

In [11]:
# Sort the sample names as you wish
sample_order = ['Neg Control', "Parental", 'LOW_uptake', 'HIGH_uptake']
# During the sorting, sorting the order of the values too. In this experiment, 
# the order of the hours should be considered too. 
# Sort based on your custom sample order and the order of hours (ascending)
df_grouped = cg.sample_order_sorter(
    df_grouped, sample_order, "Sample", ["Hours"]
)
df_grouped["Sample"].unique()

array(['Neg Control', 'Parental', 'LOW_uptake', 'HIGH_uptake'],
      dtype=object)

<div style="font-size: 0.6em; max-width: 200ch">

# Lineplot showing the cell growth over time_ covered cell area

</div>

In [12]:
df_grouped.head()

,Sample,Timepoint,Cells Selected - Number of Objects,Cells Selected - Cell Area [µm²] - Mean per Well,Cells Selected - Cell Roundness - Mean per Well,Cells Selected - Intensity Cell Digital Phase Contrast Mean - Mean per Well,Image Region - Image Region Area [µm²] - Sum per Well,Texture A Selected - Region Area [µm²] - Sum per Well,Number of Analyzed Fields,Time [s],Hours,cell_covered_area_std,cell_count_std,Samples_order
28,Neg Control,0,825.5,312.734245,0.884203,2039.592899,1.044730e+07,7.348284e+05,25.0,0.0000,4.0,47257.278319,78.488853,1
29,Neg Control,1,837.5,338.889273,0.893481,2067.363578,1.044693e+07,8.247876e+05,25.0,21564.6935,10.0,88177.844538,3.535534,1
30,Neg Control,2,868.5,386.446302,0.900810,1989.935957,1.044721e+07,9.380410e+05,25.0,43127.1585,16.0,113061.537812,24.748737,1
31,Neg Control,3,1014.0,415.559236,0.895801,1794.238645,1.044791e+07,1.080807e+06,25.0,70635.2250,24.0,90373.017414,19.798990,1
32,Neg Control,4,1226.0,416.699435,0.896083,1743.483731,1.044788e+07,1.320422e+06,25.0,91407.2465,29.0,115943.842361,42.426407,1


In [13]:
# usage 
graph = cg.Graph()
fig = graph.line_graph(
    df_grouped,
    x="Hours",  
    y="Texture A Selected - Region Area [µm²] - Sum per Well", 
    width=700,
    height=500,
)

graph.update_parameters(
    yaxis={
        "title_text": "Plate-well surface area covered by cell",
    },
    layout={
        "title": {"text": "Cell growth over time for "
                    "low-uptake and high-uptake_Exp. 4"},
        "font": {"size": 15},
      

    },
)
fig.show()

# Export the graph
# fig.write_html(r"YOUR_EXPORT_PATH\figure_title.html")

<div style="font-size: 0.6em; max-width: 100ch">

# Ratio of cell growth_ based on cell covered area

</div>

In [14]:
graph = cg.Graph()
fig = graph.line_graph(
    df_grouped,
    x="Hours",  
    y="Texture A Selected - Region Area [µm²] - Sum per Well", 
    width=700,
    height=500,
    y_axis_ratios=True
)

graph.update_parameters(
    yaxis={
        "title_text": "Growth ratios (Compared to timepoint 0)",
    },
    layout={
        "title": {
            "text": "Growth ratio over time_based on "
                "cell covered area_Exp. 4"
        },
        "font": {"size": 15},
      

    },
)
fig.show()

<div style="font-size: 0.6em; max-width: 100ch">

# Lineplot showing the cell growth over time_ absolute cell count using DPC

</div>

In [15]:
# usage 
graph = cg.Graph()
fig = graph.line_graph(
    df_grouped,
    x="Hours",  
    y="Cells Selected - Number of Objects", 
    width=700,
    height=500,
)

graph.update_parameters(
    yaxis={
        "title_text": "Plate-well surface area covered by cell",
    },
    layout={
        "title": {"text": "Cell growth over time for "
                    "low-uptake and high-uptake_Exp. 4"},
        "font": {"size": 15},
      

    },
)
fig.show()

# Export the graph
# fig.write_html(r"YOUR_EXPORT_PATH\figure_title.html")